# Day 1: Your First AI Program


**Objective:** call a model, read the reply object, check its cost, and see why the same question can give two different answers.
Every model used in this notebook ends in `:free`.

- The model used in this notebook is 'nvidia/nemotron-3-super-120b-a12b:free'.

- The below model is not used as it's endpoint is temporarily down on Oct 18,2026.
- The model that I preferred originally was 'nvidia/nemotron-3-ultra-550b-a55b:free'.
- This model is chosen based as it is the model with the most token count(Tokens processed in last 7 days in openrouter) with 6.39T tokens.
- This model is launched on Jun4, 2026. 

## Setup

1. `uv sync` from the project folder.
2. Add a `.env` file next to this notebook:

```
OPENROUTER_API_KEY=sk-or-...your key...
```

Free key: [openrouter.ai/keys](https://openrouter.ai/keys)

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# import os
# os.getenv('OPENROUTER_API_KEY')

`load_dotenv()` loads `.env` into the environment. `True` means it was found; `False` means it's missing, misnamed, or in the wrong folder.

The key is never typed into a cell or printed.

## 1. Connect to a model with ChatOpenRouter

- The entire langchain ecosystem is built on the fundamental library called `langchain_core`
- `langchain` and `langgraph` frameworks are built on top of `langchain_core` and any integrations to these frameworks by third party organisations like OpenAI(`langchain-openai`), Anthropic(`langchain-anthropic`), Google(`langchain-google-genai`), AWS(`langchain-aws`) or OpenRouter(`langchain-openrouter`) are also built by extending from this `langchain_core`. Look at https://reference.langchain.com/python/integrations/overview or search for "Langchain Python integrations"

In [5]:
from langchain_openrouter import ChatOpenRouter

# model = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free")
model = ChatOpenRouter(model="nvidia/nemotron-3-super-120b-a12b:free")

In [6]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, profile={'name': 'Nemotron 3 Super (free)', 'release_date': '2026-03-11', 'last_updated': '2026-03-11', 'open_weights': True, 'max_input_tokens': 262144, 'max_output_tokens': 262144, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x00000273AD156240>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-super-120b-a12b:free', model_kwargs={})

- `ChatOpenRouter` is a **connection**, not the model itself. Nothing is sent yet.
- The model string is `maker/model:free`.

## 2. Send your first message with invoke

In [7]:
reply = model.invoke("What is LangChain, in two sentences?")

reply.text

'LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs) by providing tools for chaining together LLMs, prompts, data sources, and other utilities. It enables developers to build complex, context-aware AI applications—like chatbots, agents, or question-answering systems—through modular and composable components.'

`invoke` sends the text and returns the reply. That's your first AI program.

## 3. The reply is an object, not a string

In [8]:
reply

AIMessage(content='LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs) by providing tools for chaining together LLMs, prompts, data sources, and other utilities. It enables developers to build complex, context-aware AI applications—like chatbots, agents, or question-answering systems—through modular and composable components.', additional_kwargs={'reasoning_content': 'Okay, the user is asking for a two-sentence definition of LangChain. They want it concise, so I need to distill the essence without fluff. \n\nHmm, judging by the query\'s simplicity, they\'re likely either a developer new to AI tools or a non-technical stakeholder trying to grasp the concept quickly. The strict sentence limit suggests they value efficiency—maybe they\'re skimming for research or comparing frameworks. \n\nI should emphasize LangChain\'s dual role: as a framework for LLM application development and its focus on composability (chains, agents,

An `AIMessage`, covered in Day 2. It has two ways to read the answer:

In [9]:
reply.text

'LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs) by providing tools for chaining together LLMs, prompts, data sources, and other utilities. It enables developers to build complex, context-aware AI applications—like chatbots, agents, or question-answering systems—through modular and composable components.'

In [10]:
reply.content

'LangChain is a framework designed to simplify the development of applications powered by large language models (LLMs) by providing tools for chaining together LLMs, prompts, data sources, and other utilities. It enables developers to build complex, context-aware AI applications—like chatbots, agents, or question-answering systems—through modular and composable components.'

Same for plain text. They diverge once a reply has images or tool calls, hence `.text`, which always gives just the words.

## 4. Count the tokens with usage_metadata

In [11]:
reply.usage_metadata

{'input_tokens': 25,
 'output_tokens': 336,
 'total_tokens': 361,
 'input_token_details': {'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'reasoning': 259}}

- `input_tokens` sent, `output_tokens` returned. A **token** ≈ ¾ of a word.
- Look inside `output_token_details`:

In [12]:
reply.usage_metadata["output_token_details"]

{'reasoning': 259}

### Reasoning tokens: paid for, never shown

`reasoning` tokens are spent **thinking**, before the answer. Not in `reply.text`, not readable, and on a paid model, billed anyway. Most current models do this.

## 5. Read response_metadata: model, finish reason, cost

In [13]:
reply.response_metadata

{'model_name': 'nvidia/nemotron-3-super-120b-a12b:free',
 'id': 'gen-1792265293-ziDAIzUDRTQpJMnBHwZg',
 'created': 1792265293,
 'object': 'chat.completion',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'openrouter',
 'cost': 0.0,
 'cost_details': {'upstream_inference_completions_cost': 0.0,
  'upstream_inference_prompt_cost': 0.0,
  'upstream_inference_cost': 0.0}}

- `model_name`: which model actually answered. Free models can be swapped or retired without notice.
- `finish_reason`: `stop` means it finished naturally (more in section 8).
- `cost`: `0.0`, proof the `:free` suffix worked.

## 6. Why the same question gives a different answer

Run the next cell two or three times.

In [16]:
model.invoke("Give me one interesting fact about the moon.").text

'One interesting fact: The Moon is drifting away from Earth at about **3.8\u202fcentimeters (1.5\u202finches) per year** due to tidal interactions, so in the distant future it will appear smaller in our sky and total solar eclipses will eventually cease.'

In [ ]:
# "One fascinating fact about the Moon is that **we only ever see one side of it from Earth** due to a phenomenon called *tidal locking* (or synchronous rotation). The Moon takes approximately the same amount of time to rotate once on its axis (~27.3 days) as it does to orbit Earth once (~27.3 days). This synchronization means the same lunar hemisphere perpetually faces us, while the "far side" (often inaccurately called the "dark side") remains hidden from direct view. Humanity didn’t glimpse the far side until 1959, when the Soviet Luna 3 spacecraft photographed it. This locking occurred over billions of years as Earth’s gravity gradually slowed the Moon’s rotation until it matched its orbital period—a elegant dance of celestial mechanics. 🌕 *(Note: Due to subtle orbital wobbles called libration, we actually see about 59% of the Moon’s surface over time, but the bulk of the far side was unknown until the space age.)*"
# "The Moon is slowly moving away from Earth at a rate of about 3.8\u202fcm per year, meaning that millions of years from now it will appear smaller in our sky and total solar eclipses will eventually cease to occur."
# "One interesting fact: The Moon is drifting away from Earth at about **3.8\u202fcentimeters (1.5\u202finches) per year** due to tidal interactions, so in the distant future it will appear smaller in our sky and total solar eclipses will eventually cease."

The answers differ. A model predicts the next word with deliberate randomness, it doesn't look one up.

**Same question, different answer** is normal, not a bug.

## 7. Control randomness with temperature

`temperature` controls how adventurous the next word choice is. `0` is as repeatable as it gets.

In [19]:
steady = ChatOpenRouter(model="nvidia/nemotron-3-super-120b-a12b:free", temperature=0)

steady.invoke("Name the largest planet in our solar system.").text

'Jupiter is the largest planet in our solar system.'

Run twice: the answer should be the same or very close. Use `temperature=0` for facts and tests, higher for ideas and variety.

Even at zero, runs aren't guaranteed identical. Treat it as "as steady as it gets," not a promise.

- Trial1: "Jupiter is the largest planet in our solar system."
- Trial2: "Jupiter is the largest planet in our solar system."

## 8. Capping the reply length with max_tokens

`max_tokens` caps the reply length. Set it absurdly low and watch what happens.

In [20]:
clipped = ChatOpenRouter(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0,
    max_tokens=20,
)

answer = clipped.invoke("Explain how a jet engine works.")

answer.text

''

In [21]:
answer

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user asked me to explain how a jet engine works. This seems like a straightforward request', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Okay, the user asked me to explain how a jet engine works. This seems like a straightforward request'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-super-120b-a12b:free', 'id': 'gen-1792265978-GBE0Pk0lQVMUvMAP7H1S', 'created': 1792265978, 'object': 'chat.completion', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a14b60-8514-7e41-8d04-7fff6ce644b0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 20, 'total_tokens': 44, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_to

Empty, not a half-finished sentence. Nothing at all.

In [22]:
answer.response_metadata["finish_reason"]

'length'

In [23]:
answer.usage_metadata

{'input_tokens': 24,
 'output_tokens': 20,
 'total_tokens': 44,
 'input_token_details': {'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'reasoning': 20}}

`finish_reason` is `length`, it hit the ceiling. The token budget went entirely to **reasoning**, the model never got to writing.

You paid in full for an empty string. Check `finish_reason`: an empty `reply.text` isn't always an error or a refusal, sometimes it's just too low a ceiling.